In [ ]:
import os
import json
import asyncio
import glob
from openai import OpenAI
from pathlib import Path

# configurations
OPENROUTER_API_KEY = "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"  
MODELS = [
    "deepseek/deepseek-v4-flash",           # think mode default, temperature=0
    "qwen/qwen3.5-flash",                  # think mode default, temperature=0
    "openai/gpt-oss-120b"           # temperature=0 (if supported)
]
FOLDER_PATH = r"E:\TUB\WS25-26\fiz\metadata_extraction\readme_files"  # folder containing README markdown files
MAX_CONCURRENT = 20                       # request concurrency limit (adjust based on your system and API limits)
OUTPUT_DIR = "."                          # output directory for JSON results

# Prompt
PROMPT = """You are a metadata extraction system for artifacts documented on GitHub.

Your task is to read the provided artifact document and extract all useful metadata that would help someone understand, index, search, reproduce, or reuse the artifact.

The artifact document is a GitHub README file in Markdown format. Use its structure (headers, code blocks, badges, lists) to help you locate and delimit evidence spans and populate evidence_location — for example, citing a Markdown header as the location, or treating a fenced code block as a single coherent span. This is a structural note only: it does not imply a fixed list of expected fields, and you should still identify whatever metadata is available in this specific document, not a generic README template.

Use only the information explicitly stated in the document. Do not use outside knowledge. Do not infer facts that are not directly supported by the text.

You may decide by yourself which metadata fields are relevant and how to name them, based entirely on what this specific document contains. Do not restrict yourself to any fixed list of field types, and do not assume the artifact belongs to any particular kind of project (e.g. software library, machine learning model, dataset, research paper, hardware project, documentation site, etc.) unless the document itself establishes that. Name each field using whatever term best describes what you found. Prefer short, snake_case field names (e.g. "programming_language", not "the language used").

For each metadata item, return:
- field: a short snake_case name for the metadata type you identified
- value: the extracted value, as close to verbatim as possible
- normalized_value: apply ONLY these normalizations, nothing else:
  - trim surrounding whitespace/punctuation
  - collapse internal whitespace
  - do not shorten, expand, or abbreviate names (e.g. keep "MIT License" as "MIT License", not "MIT"; keep "Node.js 18" as-is, not "Node 18")
  If no normalization applies, repeat value exactly.
- evidence: a text span copied VERBATIM (character-for-character) from the artifact text, including original Markdown syntax (e.g. keep "## Installation" as-is, keep code fence contents as-is). Do not paraphrase, reflow, merge non-adjacent sentences, or fix typos in the evidence field. This field will be programmatically checked as a substring of the source document.
- evidence_location: line number, Markdown section/header title, or "unknown" if not available
- confidence: one of "high", "medium", "low", using these criteria:
  - high: the value is named explicitly and unambiguously, with no plausible alternative reading
  - medium: the value is supported but requires minor interpretation (e.g. resolving an acronym used elsewhere in the doc, or a value split across adjacent sentences)
  - low: the text is genuinely ambiguous between two or more readings, or the evidence is indirect/implicit
- reason: a short explanation of why this value is useful metadata

Important rules:
1. Extract only metadata supported by evidence in the artifact text.
2. If a possible metadata value is ambiguous, include it with confidence "low" and explain the ambiguity in "reason".
3. Do not output generic concepts unless the document names a concrete value.
4. Do not invent missing metadata.
5. If the same underlying fact could be tagged with more than one field name, choose the single field name you judge most precise, and do not emit duplicate items for the same evidence span under multiple field names.
6. If no useful metadata is found, return an empty list.
7. Output ONLY the JSON object below. No markdown code fences, no preamble, no explanation outside the JSON structure.

Return only valid JSON in the following format:

{
  "metadata": [
    {
      "field": "...",
      "value": "...",
      "normalized_value": "...",
      "evidence": "...",
      "evidence_location": "...",
      "confidence": "high | medium | low",
      "reason": "..."
    }
  ]
}

Input text:
{input_text}
"""

# metadata extraction function 
def extract_metadata(client: OpenAI, model: str, doc_name: Path):
    # with open(doc_name, "r", encoding="utf-8") as f:
    #     doc_content = f.read()
    doc_content = doc_name.read_text(encoding="utf-8")
    # messages = [
    #     {"role": "system", "content": SYSTEM_PROMPT},
    #     {"role": "user", "content": f"Please extract metadata from the following GitHub README document. The artifact_id should be inferred from the project name (use the repository name if available, or the title).\n\n---\n{doc_content}\n---"}
    # ]
    messages = [
        {"role": "user", "content": PROMPT.format(input_text=doc_content)}
    ]
    # try:    
    #     extra_body = {}
    #     if "deepseek" in model.lower() or "qwen" in model.lower():
    #         extra_body["reasoning"] = {"effort": "high"}   

    #     completion = client.chat.completions.create(
    #         model=model,
    #         messages=messages,
    #         temperature=0,
    #         extra_body=extra_body if extra_body else None
    #     )
    #     raw = completion.choices[0].message.content
    #     json_str = raw.strip()
    #     if json_str.startswith("```json"):
    #         json_str = json_str.split("```json")[1].split("```")[0].strip()
    #     elif json_str.startswith("```"):
    #         json_str = json_str.split("```")[1].split("```")[0].strip()
    #     result = json.loads(json_str)
    #     return {"model": model, "doc": doc_name, "success": True, "result": result}
    # except Exception as e:
    #     return {"model": model, "doc": doc_name, "success": False, "error": str(e), "raw_response": raw if 'raw' in locals() else None}

# main async function
def main():
    # 1. read all markdown files in the folder
    md_files = sorted([x for x in Path("results/readme_files").glob("*.md")])
    # all_docs = {}
    # for file_path in md_files:
    #     doc_name = os.path.basename(file_path)
    #     with open(file_path, "r", encoding="utf-8") as f:
    #         all_docs[doc_name] = f.read()
    print(f"Found {len(md_files)} markdown files")

    # 2. initialize AsyncOpenAI client and semaphore for concurrency control
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        timeout=90.0,     
    )
    # sem = asyncio.Semaphore(MAX_CONCURRENT)

    # async def limited_extract(model, content, doc_name):
    #     async with sem:
    #         return await extract_metadata(client, model, content, doc_name)

    # # 3. construct tasks for all docs and models, skipping already processed
    # tasks = []
    # skipped_count = 0
    # for doc_name, content in all_docs.items():
    #     base_name = doc_name.replace('.md', '')
    #     for model in MODELS:
    #         model_safe = model.replace('/', '_')
    #         output_file = os.path.join(OUTPUT_DIR, f"{base_name}_metadata_{model_safe}.json")
    #         skip = False
    #         if os.path.exists(output_file):
    #             try:
    #                 with open(output_file, 'r', encoding='utf-8') as f:
    #                     data = json.load(f)
    #                     if data.get('success', False):
    #                         skip = True
    #             except:
    #                 pass
    #         if skip:
    #             skipped_count += 1
    #             print(f"Skipping {doc_name} / {model} (already processed)")
    #         else:
    #             tasks.append(limited_extract(model, content, doc_name))

    # if not tasks:
    #     print("All documents for all models have been processed. Exiting.")
    #     return

    # print(f"Processing {len(tasks)} tasks (skipped {skipped_count})")

    # # 4. execute tasks concurrently
    # all_results = await asyncio.gather(*tasks)
    # print("All requests completed.")

    # # 5. save results to individual JSON files
    # for res in all_results:
    #     doc_name = res['doc']
    #     model = res['model']
    #     base_name = doc_name.replace('.md', '')
    #     model_safe = model.replace('/', '_')
    #     output_file = os.path.join(OUTPUT_DIR, f"{base_name}_metadata_{model_safe}.json")
    #     with open(output_file, "w", encoding="utf-8") as f:
    #         json.dump(res, f, ensure_ascii=False, indent=2)
    #     print(f"Saved {output_file}")

    # # 6. failed requests summary
    # failed = [r for r in all_results if not r['success']]
    # if failed:
    #     print(f"\n{len(failed)} requests failed.")
    #     for f in failed:
    #         print(f"  - {f['doc']} / {f['model']}: {f.get('error', 'unknown error')}")
    # else:
    #     print("\nAll requests succeeded!")

# run the main function
# in jupyter notebook, use the following line:
# await main()

# python script execution (uncomment if running as a script)``
# if __name__ == "__main__":
#     asyncio.run(main())

main()

TypeError: object of type 'map' has no len()